In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
np.random.seed(42)
n_samples = 1000

data = {
    'Gender': np.random.choice(['Male', 'Female', np.nan], size=n_samples, p=[0.6, 0.38, 0.02]),
    'Age': np.random.choice([np.random.randint(18, 70) for _ in range(n_samples)], size=n_samples),
    'Debt': np.random.uniform(0, 30, size=n_samples),
    'Married': np.random.choice(['Married', 'Single', 'Divorced'], size=n_samples, p=[0.5, 0.4, 0.1]),
    'BankCustomer': np.random.choice(['g', 'p', 'gg'], size=n_samples),
    'Industry': np.random.choice(['Utilities', 'Health', 'Finance', 'Energy', 'InfoSys'], size=n_samples),
    'YearsEmployed': np.random.uniform(0, 15, size=n_samples),
    'PriorDefault': np.random.choice(['t', 'f'], size=n_samples, p=[0.4, 0.6]),
    'Employed': np.random.choice(['t', 'f'], size=n_samples, p=[0.5, 0.5]),
    'CreditScore': np.random.randint(0, 20, size=n_samples),
    'DriversLicense': np.random.choice(['t', 'f'], size=n_samples),
    'Income': np.random.exponential(scale=5000, size=n_samples),
    'Approved': np.random.choice([1, 0], size=n_samples, p=[0.45, 0.55])  # Target Variable
}

df = pd.DataFrame(data)

# Inject random NaNs into the numerical 'Age' column for pre-processing demonstration
df.loc[df.sample(frac=0.02).index, 'Age'] = np.nan

In [3]:
X = df.drop(columns=['Approved'])
y = df['Approved']

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("--- Dataset Initialized ---")
print(f"Features layout: {len(numeric_features)} numerical, {len(categorical_features)} categorical columns.\n")

--- Dataset Initialized ---
Features layout: 4 numerical, 7 categorical columns.



In [4]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Strategy for categorical: Fill missing values with most frequent string, then One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combine pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [5]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])